# OUEM D6-B — one experiment across Notebook sessions

Preparation template. Run only after review and separate owner authorization for this exact run, private inputs and free GPU runtime. Use a fresh runtime for each session. The first successful training segment is part of the 700-update experiment. See the D6-B runbook for final capacity, retention, session boundary, manual archive and scientific criteria.

In [ ]:
from pathlib import Path
import hashlib, shutil, subprocess, sys, time
from google.colab import drive

AUTHORIZED = False
CODE = Path('/content/ouem-d6-code')  # authorized private Git checkout, already staged
COMMIT = '<reviewed full commit>'
RUN_ID = 'd6-a5-main700-01'
SESSION = 'session-a'  # choose a new name after every actual runtime reset
SCRATCH = Path('/content') / ('ouem-d6-' + SESSION)
PROJECT = Path('/content/drive/MyDrive') / '<existing private OUEM folder>'
INPUTS = PROJECT / '<authorized input folder>'
RELAY_RUN = PROJECT / '<authorized existing relay parent>' / RUN_ID
ZIP_NAME = '<private A5 and Val18 ZIP filename>'
ZIP_SHA = '<expected ZIP SHA-256>'
BUNDLE_SHA = '<expected extracted bundle.json SHA-256>'
PLAN_NAME = '<approved run-plan JSON filename>'
PLAN_SHA = '<expected plan SHA-256>'
WEIGHTS_NAME = '<approved initializer filename>'
WEIGHTS_SHA = 'f8abf21cadaab2a71859e9d172bec4e5d6b7cc9710882a2f6a4d8aae826db036'
UPDATES, SECONDS, RESERVE, VRAM = 175, 1800, 300, 12 * 1024**3  # provisional
RESUME = None  # or (prior checkpoint directory's private Drive ID, expected receipt SHA)
assert AUTHORIZED, 'Owner authorization and pins required before running cells'
drive.mount('/content/drive')
assert INPUTS.is_dir() and RELAY_RUN.is_dir()

## Verify environment and free space

Resolve missing packages before training while retaining Colab's existing CUDA Torch installation. Inspect package conflicts relevant to model or raster loading. Additional dependencies may require owner-approved setup. A session with different training software fails restore. Record the Drive free-space result and persistent storage used by the two prior points, new uploads, and evidence.

In [ ]:
from importlib.metadata import version
import torch, torchvision, rasterio, numpy, PIL, pyproj, satlaspretrain_models
import google.auth
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

def git(*args):
    return subprocess.check_output(['git','-C',str(CODE),*args],text=True).strip()
def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()
def command(*args):
    return subprocess.run([sys.executable,'-B',*map(str,args)],cwd=CODE,check=True)

assert git('rev-parse','HEAD') == COMMIT and not git('status','--porcelain')
assert version('satlaspretrain-models') == '0.3.1'
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
print(torch.__version__, torchvision.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
print('scratch free bytes:', shutil.disk_usage('/content').free)
assert shutil.disk_usage('/content').free >= 6 * 1024**3, 'Reassess scratch capacity'

# Independent remote readback uses Drive API bytes, separate from the FUSE mount.
auth.authenticate_user()
credentials,_ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
api = build('drive','v3',credentials=credentials,cache_discovery=False)
quota = api.about().get(fields='storageQuota').execute()['storageQuota']
print('Drive quota record:', quota)
REQUIRED_DRIVE_FREE = 4_000_000_000  # provisional, adjust after observed checkpoint sizes
MANUAL_FREE_BYTES = None  # record a separately verified quota if the API omits limit
free = (int(quota['limit'])-int(quota['usage'])) if 'limit' in quota and 'usage' in quota else MANUAL_FREE_BYTES
assert free is not None and free >= REQUIRED_DRIVE_FREE

## Stage once into this runtime

The bundle contains the exact A5 shards and historical Val18, plus relocation information. Keep downloads outside the Git checkout and revalidate from scratch. Session B must have a different `SESSION` and a fresh `SCRATCH` after discarding session A's runtime.

In [ ]:
SCRATCH.mkdir()
for name,digest in [(ZIP_NAME, ZIP_SHA), (PLAN_NAME, PLAN_SHA), (WEIGHTS_NAME, WEIGHTS_SHA)]:
    assert Path(name).name == name
    t = time.perf_counter()
    with (INPUTS/name).open('rb') as src,(SCRATCH/name).open('xb') as dst:
        shutil.copyfileobj(src,dst,1024**2)
    assert sha(SCRATCH/name)==digest
    print(name, (SCRATCH/name).stat().st_size, 'bytes;', round(time.perf_counter()-t,2), 's staged')
command('scripts/d6b_bundle.py','unpack','--archive',SCRATCH/ZIP_NAME,
        '--sha256',ZIP_SHA,'--output',SCRATCH/'dataset')
assert sha(SCRATCH/'dataset/bundle.json')==BUNDLE_SHA

## Fetch and verify a committed point from Drive

Supply the private folder ID of the checkpoint directory and the receipt SHA recorded independently from local CLI output. The API downloads receipt and files by their Drive IDs into an empty directory. This is also used after the segment for readback. Use the last confirmed point. Record any later uncommitted updates as lost work.

In [ ]:
import json, re

def remote_children(folder_id):
    assert re.fullmatch(r'[A-Za-z0-9_-]+',folder_id)
    records, token = [], None
    while True:
        result = api.files().list(q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken,files(id,name,mimeType,size)',pageToken=token).execute()
        records.extend(result['files'])
        token = result.get('nextPageToken')
        if not token:
            break
    result={r['name']:r for r in records}
    assert len(result)==len(records), 'Ambiguous Drive filenames'
    return result

def from_drive(item,dest):
    assert int(item['size']) >= 0
    with dest.open('xb') as output:
        dl = MediaIoBaseDownload(output,api.files().get_media(fileId=item['id']),chunksize=16*1024**2)
        done = False
        while not done:
            _,done = dl.next_chunk(num_retries=0)
    assert dest.stat().st_size == int(item['size'])

def remote_verify(folder_id,dest,pin):
    t=time.perf_counter()
    dest.mkdir()
    children=remote_children(folder_id)
    from_drive(children['receipt.json'],dest/'receipt.json')
    assert sha(dest/'receipt.json')==pin
    receipt=json.loads((dest/'receipt.json').read_bytes())
    listings={}
    for rel,entry in receipt['files'].items():
        parts=rel.split('/')
        assert all(x and x not in ('.','..') and ':' not in x and chr(92) not in x for x in parts)
        current=children
        for component in parts[:-1]:
            sub=current[component]
            assert sub['mimeType']=='application/vnd.google-apps.folder'
            if sub['id'] not in listings:
                listings[sub['id']]=remote_children(sub['id'])
            current=listings[sub['id']]
        item=current[parts[-1]]
        assert int(item['size'])==entry['size']
        target=dest.joinpath(*parts)
        target.parent.mkdir(parents=True,exist_ok=True)
        from_drive(item,target)
        assert sha(target)==entry['sha256']
    command('scripts/d6b_relay.py',dest,'--receipt-sha256',pin)
    print('REMOTE_READBACK_PASS',pin,round(time.perf_counter()-t,2),'s')

if RESUME:
    remote_verify(RESUME[0],SCRATCH/'resumed',RESUME[1])

## Execute one bounded segment of the same run

Use the same plan, commit, initializer and data pins across sessions. The trainer commits checkpoints at cadence and safe segment boundaries and copies evaluation and update evidence into the corresponding relay record. At zero progress or error inspect the existing receipts. Avoid restarting from an unconfirmed or stale mount copy.

In [ ]:
plan = json.loads((SCRATCH/PLAN_NAME).read_bytes())
assert plan['run_id']==RUN_ID
args=['scripts/d6b_train.py','--bundle',SCRATCH/'dataset','--bundle-sha256',BUNDLE_SHA,
      'segment','--plan',SCRATCH/PLAN_NAME,'--plan-sha256',PLAN_SHA,
      '--initializer',SCRATCH/WEIGHTS_NAME,'--output',SCRATCH/'run','--backup',RELAY_RUN/SESSION,
      '--session',SESSION,'--updates',UPDATES,'--seconds',SECONDS,'--reserve',RESERVE,'--max-vram',VRAM]
if RESUME:
    args+=['--resume',SCRATCH/'resumed','--receipt-sha256',RESUME[1]]
try:
    command(*args)
finally:
    for p in sorted((SCRATCH/'run').glob('*/receipt.json')):
        print(p.parent.name, sha(p))  # retain these pins independently of the Drive mount

## Verify the relay before disconnect and archive on GEEKOM-A6

Use the private Drive folder ID for this session (visible in the existing folder URL without changing sharing). Reopen all committed records through the API. Confirm Step 0 evidence as well as checkpoints and the segment summary. Record the checkpoint and receipt pins for session B. After confirming a recoverable point, disconnect and delete the Notebook runtime to discard scratch, then use a new GPU runtime when allowed. Keep the previous known-good point. For GEEKOM download by the existing private route, then run the archive verification/copy from the runbook before considering any owner-controlled cleanup.

In [ ]:
SESSION_REMOTE_ID = '<private Drive ID of the completed session directory>'
records=remote_children(SESSION_REMOTE_ID)
for p in sorted((SCRATCH/'run').glob('*/receipt.json')):
    name=p.parent.name
    remote_verify(records[name]['id'],SCRATCH/('remote-'+name),sha(p))

## Final comparison

After the final checkpoint at the frozen update budget, compare each cohort against its own Step 0 from the verified receipts. Store the comparison JSON and environment/timing receipts with the run on Drive and GEEKOM. Milestone success requires at least two actually distinct GPU runtimes with independently verified remote restore, full update coverage, and durable archival. Accuracy improvement is not a success threshold.

In [ ]:
sys.path.insert(0,str(CODE/'scripts'))
from d6b_train import compare_cohorts, json_new
from d6b_relay import seal,copy

FINALIZE = False
STEP0_ID = '<private Step 0 evidence folder ID>'
STEP0_PIN = '<recorded Step 0 receipt SHA>'
FINAL_DIR = SCRATCH/'remote-checkpoint-0700'
FINAL_PIN = '<recorded final receipt SHA>'
if FINALIZE:
    baseline = SCRATCH/'comparison-baseline'
    remote_verify(STEP0_ID,baseline,STEP0_PIN)
    report = SCRATCH/'comparison'
    report.mkdir()
    result=compare_cohorts(baseline,STEP0_PIN,FINAL_DIR,FINAL_PIN)
    json_new(report/'comparison.json',result)
    pin=seal(report,dict(kind='comparison',run_id=RUN_ID,session=SESSION,
                         initial_receipt_sha256=STEP0_PIN,final_receipt_sha256=FINAL_PIN))
    copy(report,pin,RELAY_RUN/SESSION/'comparison')
    print('Comparison receipt SHA:',pin)
    # Verify this new relay directory separately with remote_verify using its Drive ID.
